In [64]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib as plt

print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"matplotlib version: {plt.__version__}")

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
NumPy version: 2.0.2
pandas version: 2.3.3
scikit-learn version: 1.6.1
matplotlib version: 3.9.4


## Step2 Load and inspect

In [65]:
import pandas as pd

raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=3132)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

(1158, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           60
building_type         0
ceiling_height_m    136
price_kzt             0
dtype: int64
                   count unique         top freq         mean         std  \
listing_id          1158   1080   ALM-10932    3          NaN         NaN   
listed_date         1158    246  2026-04-18   12          NaN         NaN   
district            1158     47       Medeu  147          NaN         NaN   
rooms               1158      6           2  440          NaN         NaN   
area_

In [66]:
print("Sample values of potential numeric columns")

for col in ["listed_date", "area_m2", "rooms", "floor", "total_floors","year_built", "ceiling_height_m", "price_kzt"]:
    if col in df.columns and df[col].dtype == "object":
        print(f"\n{col} samples:")
        print(df[col].sample(10, random_state=3132).tolist())
    

Sample values of potential numeric columns

listed_date samples:
['2026-02-23', '2026-01-30', '2026-01-31', '2026-05-18', '2026-01-27', '2026-02-10', '2026-04-25', '2026-03-11', '2026-07-11', '2026-05-29']

area_m2 samples:
['53.0', '105.9', '112.0 m²', '60.9', '58.8', '77.4', '57.8', '81.0', '77.5', '71.2']

rooms samples:
['2', '4', '4', '2', '2', '3', '2', '3', '3', '3']

price_kzt samples:
['35120000', '47130000', '125100000', '39190000', '59850000', '37960000', '47940000', '89 740 000 ₸', '48,480,000', '71100000']


In [67]:
print("Count duplicates")

exact_dups = df.duplicated().sum()
listing_dups = df.drop_duplicates()["listing_id"].duplicated().sum()

print(f"\n(a) Exact duplicate rows: {exact_dups}")
print(f"(b) Non-exact duplicates with same listing_id: {listing_dups}")

dup_listing_ids = df[df.duplicated(subset=["listing_id"], keep=False)]["listing_id"].drop_duplicates().iloc[:2]
print("\n--- Duplicate listing examples ---")
print(df[df["listing_id"].isin(dup_listing_ids)].sort_values("listing_id"))

Count duplicates

(a) Exact duplicate rows: 35
(b) Non-exact duplicates with same listing_id: 43

--- Duplicate listing examples ---
    listing_id listed_date district rooms area_m2  floor  total_floors  \
8    ALM-10003  2026-05-29   Auezov     2    48.6      2             5   
823  ALM-10003  2026-05-29   Auezov     2    48.6      2             5   
967  ALM-10003  2026-05-04   Auezov     2    48.6      2             5   
12   ALM-10938  2026-08-10    Medeu     5   133.0      1             9   
509  ALM-10938  2026-08-21    Medeu     5   133.0      1             9   

     year_built building_type  ceiling_height_m  price_kzt  
8        1978.0         panel              2.55   26120000  
823      1978.0         panel              2.55   26120000  
967      1978.0         panel              2.55   28320000  
12       2013.0      monolith              2.93  177470000  
509      2013.0      monolith              2.93  167850000  


### Step 2: Data Inspection Answers

#### 1.Types
pandas don't choose because area_m2, price_kzt should can be numbers but they have another symbols like kzt m2. for that reason pandas says they object

#### 2.Grain
duplicate listing_id but other attributes like price_kzt or listed_data different ,so it means same listing was reposted or updated with a new price or time.

#### 3.Validation Spec
| Column | Expected Type | Valid Range / Allowed Values |
| :--- | :--- | :--- |
| listing_id | Integer | Positive integer |
| listed_date | Date (datetime64) | 2026-05-01 to 2026-09-28 |
| district | Categorical / String |  Almaty districts: Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu   |
| rooms | Integer | 1 to 5 |
| area_m2 | Float | 20.0 to 250.0 |
| floor | Integer | 1 to total_floors  |
| total_floors | Integer | 1 to 25 |
| year_built | Integer | 1950 to 2026 (0 is a sentinel, not a valid value) |
| building_type | Categorical / String |panel, brick, monolith |
| ceiling_height_m | Float | 2.2 to 4.0 m  |
| price_kzt | Float | 10,000,000 to 200,000,000 ₸ |

## Step 3: Clean and transform

In [68]:
cleaning_log = []

def log_step(operation_name, df_current):
    rows = len(df_current)
    cleaning_log.append({"Operation": operation_name, "Rows": rows})
    print(f"[{operation_name}] Rows count: {rows}")

log_step("Initial state (Step 2 output)", df)

[Initial state (Step 2 output)] Rows count: 1158


In [69]:
# 3a. Duplicates and Reposts
df = df.drop_duplicates().reset_index(drop=True)
log_step("3a. After dropping exact duplicates", df)

df["listed_date"] = pd.to_datetime(df["listed_date"], errors="coerce")

reposts_before = df.drop_duplicates()["listing_id"].duplicated().sum()
df = df.sort_values("listed_date").drop_duplicates(subset=["listing_id"], keep="last").reset_index(drop=True)
log_step("3a. After resolving reposts (kept latest date)", df)

print(f"\nReposts resolved: {reposts_before}")

[3a. After dropping exact duplicates] Rows count: 1123
[3a. After resolving reposts (kept latest date)] Rows count: 1080

Reposts resolved: 43


In [70]:
# 3b. District Categories Standardisation

print("\n--- Original District Values ---")
print(df["district"].value_counts(dropna=False))

def clean_district(val):
    if pd.isna(val):
        return val
    s = str(val).strip().lower()
    
    if "алатау" in s or "alatau" in s: return "Alatau"
    if "алмал" in s or "almal" in s: return "Almaly"
    if "ауэз" in s or "auez" in s: return "Auezov"
    if "бостан" in s or "bostan" in s: return "Bostandyk"
    if "медеу" in s or "medeu" in s: return "Medeu"
    if "наурыз" in s or "nauryz" in s: return "Nauryzbay"
    if "турк" in s or "turk" in s: return "Turksib"
    if "жет" in s or "zhet" in s: return "Zhetysu"
    return val

df["district"] = df["district"].apply(clean_district)
log_step("3b. After district standardization", df)

print(f"\nUnique districts count: {df['district'].nunique()}")
print(f"Missing districts count: {df['district'].isna().sum()}")
print("Districts values:")
print(df["district"].value_counts())


--- Original District Values ---
district
Medeu            136
Bostandyk        124
Zhetysu          121
Auezov           119
Turksib          113
Almaly           108
Alatau           106
Nauryzbay         93
turksib           10
medeu             10
Наурызбайский      7
auezov             7
Nauryzbayskiy      6
Alatauskiy         6
Auezovskiy         6
 Zhetysu           6
AUEZOV             6
nauryzbay          5
 Turksib           5
Bostandykskiy      5
 Auezov            5
BOSTANDYK          5
ALMALY             5
bostandyk          4
 Medeu             4
Медеуский          4
NAURYZBAY          4
 Alatau            4
TURKSIB            4
almaly             4
Zhetysuskiy        3
MEDEU              3
Турксибский        3
 Almaly            3
Бостандыкский      3
 Bostandyk         3
ZHETYSU            3
Almalinskiy        3
Medeuskiy          2
Ауэзовский         2
zhetysu            2
ALATAU             2
Turksibskiy        2
alatau             1
Алатауский         1
Алмалинский 

In [71]:
# --- Step 3c: Text to Numbers Transformation  ---

# 1. rooms:
if df["rooms"].dtype == "object":
    df["rooms"] = df["rooms"].astype(str).str.strip().str.lower()
    df["rooms"] = df["rooms"].replace({"studio": "1", "студия": "1"})
    df["rooms"] = pd.to_numeric(df["rooms"].str.replace(r"\D", "", regex=True), errors="coerce")

# 2. area_m2:
if df["area_m2"].dtype == "object":
    df["area_m2"] = df["area_m2"].astype(str).str.replace("m²", "", regex=False)
    df["area_m2"] = df["area_m2"].str.replace("m2", "", regex=False)
    df["area_m2"] = df["area_m2"].str.replace(",", ".", regex=False).str.strip()
    df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")

# 3. price_kzt:
if df["price_kzt"].dtype == "object":
    df["price_kzt"] = df["price_kzt"].astype(str).str.replace("₸", "", regex=False)
    df["price_kzt"] = df["price_kzt"].str.replace("kzt", "", regex=False, case=False)
    df["price_kzt"] = df["price_kzt"].str.replace(r"\s+", "", regex=True)
    df["price_kzt"] = df["price_kzt"].str.replace(",", "", regex=False)
    df["price_kzt"] = pd.to_numeric(df["price_kzt"], errors="coerce")

# 4. ceiling_height_m:
if "ceiling_height_m" in df.columns and df["ceiling_height_m"].dtype == "object":
    df["ceiling_height_m"] = df["ceiling_height_m"].astype(str).str.replace("m", "", regex=False)
    df["ceiling_height_m"] = df["ceiling_height_m"].str.replace(",", ".", regex=False).str.strip()
    df["ceiling_height_m"] = pd.to_numeric(df["ceiling_height_m"], errors="coerce")

log_step("3c. After converting text columns to numeric", df)

print("\n--- Final Data Types (df.dtypes) ---")
print(df.dtypes)

print("--- Missing Values Check After Fix ---")
print(df[["rooms", "area_m2", "price_kzt"]].isna().sum())

[3c. After converting text columns to numeric] Rows count: 1080

--- Final Data Types (df.dtypes) ---
listing_id                  object
listed_date         datetime64[ns]
district                    object
rooms                        int64
area_m2                    float64
floor                        int64
total_floors                 int64
year_built                 float64
building_type               object
ceiling_height_m           float64
price_kzt                    int64
dtype: object
--- Missing Values Check After Fix ---
rooms        0
area_m2      0
price_kzt    0
dtype: int64


## Step 4: Sentinels, impossible values, outliers

In [72]:
# --- Step 4a: Sentinels ---
print(df[["floor", "year_built", "ceiling_height_m"]].describe())

sentinel_cols = {
    "floor": -1,
    "year_built": 0,
    "ceiling_height_m": 0,
}

for col, sent_val in sentinel_cols.items():
    count = (df[col] == sent_val).sum()
    print(f"{col}: {count} sentinel value(s) found (= {sent_val})")
    df[col] = df[col].replace(sent_val, np.nan)

log_step("4a. After replacing sentinels with NaN", df)
print(df[["floor", "year_built", "ceiling_height_m"]].isna().sum())

             floor   year_built  ceiling_height_m
count  1080.000000  1023.000000        954.000000
mean      5.925000  1945.376344          2.713826
std       4.803934   295.862079          0.451544
min      -1.000000     0.000000          0.000000
25%       2.000000  1972.000000          2.620000
50%       5.000000  1986.000000          2.770000
75%       8.000000  2009.500000          2.910000
max      30.000000  2024.000000          3.200000
floor: 11 sentinel value(s) found (= -1)
year_built: 23 sentinel value(s) found (= 0)
ceiling_height_m: 22 sentinel value(s) found (= 0)
[4a. After replacing sentinels with NaN] Rows count: 1080
floor                11
year_built           80
ceiling_height_m    148
dtype: int64


In [73]:
# --- Step 4b: Impossible values ---

bad_rooms = ~df["rooms"].between(1, 5)
print("rooms outside spec:", bad_rooms.sum())

bad_floor = df["floor"] > df["total_floors"]
print("floor > total_floors:", bad_floor.sum())
print(df.loc[bad_floor, ["floor", "total_floors"]])

bad_year = df["year_built"] > 2026
print("year_built > 2026:", bad_year.sum())

bad_area = ~df["area_m2"].between(20, 250)
print("area_m2 outside spec:", bad_area.sum())
print(df.loc[bad_area, ["listing_id", "area_m2", "price_kzt"]])

df.loc[bad_floor, "floor"] = np.nan

log_step("4b. After marking impossible values", df)

rooms outside spec: 0
floor > total_floors: 10
     floor  total_floors
9     15.0            12
147    4.0             3
315   30.0            25
568   11.0             9
610    6.0             5
696    6.0             5
748   10.0             5
818   13.0            12
878    8.0             5
989   11.0             9
year_built > 2026: 0
area_m2 outside spec: 11
     listing_id  area_m2  price_kzt
0     ALM-10910    736.0   54050000
178   ALM-10405    356.0   18830000
214   ALM-11122    647.0   40670000
253   ALM-10072    530.0   64270000
322   ALM-10058    555.0   66270000
506   ALM-10921    668.0   38560000
648   ALM-10264    495.0   42680000
758   ALM-10909    565.0   32610000
939   ALM-10440    879.0   48370000
995   ALM-10069    481.0   27710000
1048  ALM-10659    544.0   25030000
[4b. After marking impossible values] Rows count: 1080


In [74]:
# --- Step 4c: Outliers ---
df["price_per_m2"] = df["price_kzt"] / df["area_m2"]
df["area_before_fix"] = df["area_m2"]

def fences(values, groups=None):
    grouped = values if groups is None else values.groupby(groups)
    q1, q3 = grouped.quantile(0.25), grouped.quantile(0.75)
    if groups is not None:
        q1, q3 = groups.map(q1), groups.map(q3)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

low, high = fences(df["price_per_m2"])
global_flag = (df["price_per_m2"] < low) | (df["price_per_m2"] > high)
print("Global IQR flagged:", global_flag.sum())

low_g, high_g = fences(df["price_per_m2"], df["building_type"])
group_flag = (df["price_per_m2"] < low_g) | (df["price_per_m2"] > high_g)
print("Grouped (by building_type) IQR flagged:", group_flag.sum())

Global IQR flagged: 0
Grouped (by building_type) IQR flagged: 0


In [75]:
# --- Step 4d: Decide what to do with each flagged group ---
fix_idx = df.loc[bad_area].index
print("Rows corrected (area_m2 / 10):", len(fix_idx))
df.loc[fix_idx, "area_m2"] = df.loc[fix_idx, "area_m2"] / 10

df["price_per_m2"] = df["price_kzt"] / df["area_m2"]

log_step("4d. After correcting area_m2 and deciding on flagged rows", df)

df.to_csv("data/almaty_apartments_clean.csv", index=False)
print("Saved:", df.shape)

Rows corrected (area_m2 / 10): 11
[4d. After correcting area_m2 and deciding on flagged rows] Rows count: 1080
Saved: (1080, 13)


### Step 4: Sentinels, Impossible Values, Outliers

#### Sentinels
I checked describe() and found 3 columns had a fake value instead of missing:
- floor had -1 in 11 rows (floor can't be negative)
- year_built had 0 in 23 rows (no building was built in year 0)
- ceiling_height_m had 0 in 22 rows (ceiling can't be 0m)

I replaced all of these with NaN. I didn't drop the rows because the other columns in those rows are still fine.

#### Impossible values
I checked the data against my validation spec from Step 2:
- rooms outside 1-5: none found
- year_built after 2026: none found
- floor bigger than total_floors: 10 rows. This is weird because a flat can't be on a higher floor than the building has. I couldn't tell if floor or total_floors was the wrong one, so I just set floor to NaN for these rows.
- area_m2 outside 20-250: 11 rows, with values like 736, 879, 356 etc. When I divided these by 10, they became normal numbers (like 73.6, 87.9) and matched the price level of similar apartments. Looks like someone typed the decimal point in the wrong place, so I fixed it by dividing by 10 instead of dropping the rows.

#### Outliers
I made price_per_m2 = price_kzt / area_m2 and used the 1.5*IQR rule. Both the global version and the version grouped by building_type gave 0 flagged rows. At first this looked wrong, but I think it's because the area_m2 typos I found above already got hidden inside price_per_m2 — dividing price by a much bigger area doesn't make the ratio extreme enough to cross the fence. So the IQR check on price_per_m2 didn't catch anything new, the area_m2 check above already found the real problem.

#### What I did overall
- sentinels (floor/year_built/ceiling_height) -> NaN
- floor > total_floors (10 rows) -> NaN, since I don't know which value is wrong
- area_m2 way too big (11 rows) -> divided by 10 to fix
- price_per_m2 outliers -> none found separately

Saved the cleaned data (with NaN still in it) to data/almaty_apartments_clean.csv.

## Step 5: Missing data

In [76]:
# --- Step 5a: Split first ---
from sklearn.model_selection import train_test_split

NUMERIC = ["rooms", "area_m2", "floor", "total_floors", "year_built", "ceiling_height_m"]

train, test = train_test_split(df, test_size=0.2, random_state=3132)
train, test = train.copy(), test.copy() 

print(train.shape, test.shape)
print(train[NUMERIC].isna().sum())

(864, 13) (216, 13)
rooms                 0
area_m2               0
floor                18
total_floors          0
year_built           65
ceiling_height_m    120
dtype: int64


In [77]:
# --- Step 5b: Diagnose the mechanism on the training set ---
rate_ceiling = train.assign(missing=train["ceiling_height_m"].isna()).groupby("building_type")["missing"].mean()
print("ceiling_height_m missing rate by building_type:")
print(rate_ceiling)

rate_year = train.assign(missing=train["year_built"].isna()).groupby("building_type")["missing"].mean()
print("\nyear_built missing rate by building_type:")
print(rate_year)

print("\nRows that dropna() would remove:", train[NUMERIC].isna().any(axis=1).sum())

ceiling_height_m missing rate by building_type:
building_type
brick       0.142857
monolith    0.034602
panel       0.220670
Name: missing, dtype: float64

year_built missing rate by building_type:
building_type
brick       0.092166
monolith    0.069204
panel       0.069832
Name: missing, dtype: float64

Rows that dropna() would remove: 194


In [78]:
# --- Step 5c: Compare imputation strategies with a masking experiment ---

from sklearn.metrics import mean_absolute_error

known = train[train["ceiling_height_m"].notna()]
hidden = known.sample(frac=0.15, random_state=3132).index
rest = train.drop(index=hidden)                    
answers = train.loc[hidden, "ceiling_height_m"]    
print(len(hidden), "values hidden")

# Method 1: global median
global_median = rest["ceiling_height_m"].median()
pred_global = pd.Series(global_median, index=hidden)
mae_global = mean_absolute_error(answers, pred_global)

# Method 2: median within each building_type
group_median = rest.groupby("building_type")["ceiling_height_m"].median()
pred_group = train.loc[hidden, "building_type"].map(group_median)
mae_group = mean_absolute_error(answers, pred_group)

print(f"Global median MAE: {mae_global:.4f}")
print(f"Group median MAE:  {mae_group:.4f}")

112 values hidden
Global median MAE: 0.1376
Group median MAE:  0.0785


In [79]:
# --- Step 5d: Fill ---

ceiling_med_by_group = train.groupby("building_type")["ceiling_height_m"].median()
train["ceiling_height_m"] = train["ceiling_height_m"].fillna(train["building_type"].map(ceiling_med_by_group))
test["ceiling_height_m"] = test["ceiling_height_m"].fillna(test["building_type"].map(ceiling_med_by_group))

year_med_by_group = train.groupby("building_type")["year_built"].median()
train["year_built"] = train["year_built"].fillna(train["building_type"].map(year_med_by_group))
test["year_built"] = test["year_built"].fillna(test["building_type"].map(year_med_by_group))

floor_med = train["floor"].median()
train["floor"] = train["floor"].fillna(floor_med)
test["floor"] = test["floor"].fillna(floor_med)

print("Train NaN left in NUMERIC:", train[NUMERIC].isna().sum().sum())
print("Test NaN left in NUMERIC:", test[NUMERIC].isna().sum().sum())

Train NaN left in NUMERIC: 0
Test NaN left in NUMERIC: 0


### Step 5: Missing Data — Report

- Split: train (864 rows) / test (216 rows), before any filling.
- Missing counts (train): floor 18, year_built 65, ceiling_height_m 120.
- ceiling_height_m missing rate by building_type: brick 0.14, monolith 0.03, panel 0.22
  -> rates are clearly different across groups -> **MAR** (depends on building_type).
- year_built missing rate by building_type: brick 0.09, monolith 0.07, panel 0.07
  -> rates are close to each other -> looks like **MCAR**.
- dropna() would remove 194 rows (~22% of train) - too many to just drop.
- Masking experiment on ceiling_height_m (112 values hidden):
  global median MAE = 0.1376, group median MAE = 0.0785.
  Group median is better, which matches the MAR finding above.
- Fill: ceiling_height_m and year_built filled with median per building_type
  (computed from train only); floor filled with the train's global median.
- No NaN left in NUMERIC in either train or test.